# AI-Finance: ML Training & Tournament

**Crypto Swing Trading System** — Walk-forward ML with regime filtering

## What this notebook does:
1. **Loads 4h OHLCV data** for 200+ tokens from Google Drive
2. **Builds 22+ stationary features** (no raw prices — ratios, returns, oscillators only)
3. **Runs ML Tournament** — XGBoost, LightGBM, Random Forest, Logistic Regression
4. **Walk-forward validation** with purged embargo gap (no lookahead)
5. **Expert Backtest v3** — BTC-specific with 4-gate regime filter + ATR stops
6. **Visualizes results** — equity curves, feature importance, monthly breakdown

---

In [ ]:
# ============================================================
# Cell 1: Install Dependencies
# ============================================================
!pip install -q xgboost lightgbm scikit-learn pandas numpy matplotlib seaborn

In [ ]:
# ============================================================
# Cell 2: Imports
# ============================================================
import glob
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
plt.style.use('dark_background')
sns.set_palette('bright')

print('All imports loaded successfully!')

In [ ]:
# ============================================================
# Cell 3: Mount Google Drive & Set Data Path
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# IMPORTANT: Upload your data/raw/4h folder to Google Drive first!
# Expected structure: Drive/AI-Finance/data/raw/4h/{BTCUSDT,ETHUSDT,...}/*.csv
#                     Drive/AI-Finance/data/raw/4h/manifest.csv
#
# Option A: Use Google Drive path
RAW_DIR = Path('/content/drive/MyDrive/AI-Finance/data/raw/4h')
#
# Option B: Upload directly to Colab (uncomment if not using Drive)
# from google.colab import files
# uploaded = files.upload()  # Upload manifest.csv + token CSVs
# RAW_DIR = Path('/content/data/raw/4h')

# Verify data exists
if RAW_DIR.exists():
    manifest_path = RAW_DIR / 'manifest.csv'
    if manifest_path.exists():
        manifest = pd.read_csv(manifest_path)
        print(f'Found {len(manifest)} tokens in manifest')
        print(f'Top tokens by data months:')
        print(manifest.sort_values('months', ascending=False).head(10).to_string(index=False))
    else:
        print(f'manifest.csv not found at {manifest_path}')
else:
    print(f'Data directory not found: {RAW_DIR}')
    print('Please upload your data to Google Drive first.')

In [ ]:
# ============================================================
# Cell 4: Strategy Parameters
# ============================================================

# --- Tournament Parameters ---
FEE_PCT = 0.002                # 0.2% trading fee per trade
TARGET_HORIZON = 30            # 30 x 4h = 5 days forward
TARGET_THRESHOLD = 0.02        # 2% move = positive class
RETRAIN_EVERY = 540            # ~90 days (quarterly) in 4h candles
MIN_TRAIN = 2000               # Minimum training samples
CONFIDENCE_THRESHOLD = 0.55    # Minimum predicted probability to act

# --- BTC Backtest v3 Parameters ---
BTC_TARGET_RETURN_PCT = 0.03   # Predict: BTC up >3% in 14 days
BTC_TARGET_HORIZON = 14        # days
BTC_CONFIDENCE_GATE = 0.60     # min probability to act
COOLDOWN_DAYS = 14             # min days between entries
MAX_POSITIONS = 3
ATR_MULTIPLIER = 2.0           # stop = 2x ATR
ATR_STOP_FLOOR = 0.05          # min 5% stop
ATR_STOP_CEILING = 0.10        # max 10% stop
MOMENTUM_GATE = -0.12          # reject if 20d ROC < -12%
MAX_HOLD_DAYS = 42             # force exit after 6 weeks
STAGNATION_DAYS = 21           # exit if flat after 3 weeks
STAGNATION_BAND = 0.03         # "flat" = within +/-3%
BTC_RETRAIN_EVERY = 60         # retrain every 60 days
EMBARGO_DAYS = 21              # purge gap
MIN_TRAIN_BTC = 180            # min BTC training samples

print('Parameters configured.')

In [ ]:
# ============================================================
# Cell 5: Feature Engineering (Stationary Only)
# ============================================================

def build_features_fast(df):
    """Optimized stationary feature engineering for 4h OHLCV candles.
    
    NO raw price levels — all features are ratios, returns, or oscillators.
    Works universally across 200+ tokens.
    """
    close = df['close'].astype(float)
    high = df['high'].astype(float)
    low = df['low'].astype(float)
    volume = df['volume'].astype(float)
    out = pd.DataFrame(index=df.index)

    # ── Momentum (log returns at multiple scales) ──
    for p in [6, 12, 24, 42, 84, 168]:
        out[f'ret_{p}'] = np.log(close / close.shift(p))

    # ── Price position relative to SMAs ──
    for p in [20, 50, 100, 200]:
        out[f'p2sma_{p}'] = close / close.rolling(p).mean() - 1

    # ── EMA ratios ──
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    out['ema_ratio'] = ema12 / ema26 - 1

    # ── RSI (normalized to [-1, 1]) ──
    delta = close.diff()
    gain = delta.where(delta > 0, 0.0).rolling(14).mean()
    loss = (-delta).where(delta < 0, 0.0).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    out['rsi'] = (100 - (100 / (1 + rs)) - 50) / 50

    # ── MACD histogram (normalized by price) ──
    out['macd_norm'] = ((ema12 - ema26) - (ema12 - ema26).ewm(9).mean()) / close

    # ── Bollinger Bands (stationary derivatives only) ──
    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    out['bb_width'] = (2 * bb_std) / bb_mid
    out['bb_pct'] = (close - (bb_mid - 2 * bb_std)) / (4 * bb_std)

    # ── Volatility at multiple scales ──
    ret = close.pct_change()
    out['vol_6'] = ret.rolling(6).std()
    out['vol_24'] = ret.rolling(24).std()
    out['vol_72'] = ret.rolling(72).std()
    out['vol_ratio'] = out['vol_6'] / out['vol_72'].replace(0, np.nan)

    # ── ATR normalized by price ──
    prev_c = close.shift(1)
    tr = pd.concat([high - low, (high - prev_c).abs(), (low - prev_c).abs()], axis=1).max(axis=1)
    out['atr_norm'] = tr.rolling(14).mean() / close

    # ── Volume ratio ──
    out['vol_ratio_v'] = volume / volume.rolling(20).mean().replace(0, np.nan)

    # ── Microstructure ──
    out['clv'] = (2 * close - high - low) / (high - low).replace(0, np.nan)

    # ── Distribution ──
    out['skew'] = ret.rolling(24).skew()
    out['drawdown'] = close / close.rolling(168).max() - 1

    # ── Regime indicators (binary) ──
    out['above_sma50'] = (close > close.rolling(50).mean()).astype(float)
    out['above_sma200'] = (close > close.rolling(200).mean()).astype(float)

    # ── Time features ──
    if 'open_time' in df.columns:
        # Auto-detect time unit (ms vs us)
        sample_val = df['open_time'].iloc[0]
        if sample_val > 1e15:  # microseconds
            ts = pd.to_datetime(df['open_time'], unit='us')
        elif sample_val > 1e12:  # milliseconds
            ts = pd.to_datetime(df['open_time'], unit='ms')
        else:
            ts = pd.to_datetime(df['open_time'], unit='s')
        out['hour_sin'] = np.sin(2 * np.pi * ts.dt.hour / 24)
        out['dow_sin'] = np.sin(2 * np.pi * ts.dt.dayofweek / 7)

    # ── Raw values for backtest (NOT features) ──
    out['_close'] = close
    out['_atr'] = tr.rolling(14).mean()

    return out


print('Feature engineering function defined.')
print('Features: 6 momentum + 4 SMA ratios + 1 EMA + RSI + MACD + 2 BB + 4 vol + ATR + volume + CLV + skew + drawdown + 2 regime + 2 time = 26 features')

In [ ]:
# ============================================================
# Cell 6: Load All Tokens
# ============================================================

def load_all_tokens(raw_dir, max_tokens=200):
    """Load tokens from CSV, compute features, combine."""
    manifest = pd.read_csv(raw_dir / 'manifest.csv')
    manifest = manifest.sort_values('months', ascending=False).head(max_tokens)

    all_data = []
    feature_cols = None

    for idx, (_, row) in enumerate(manifest.iterrows()):
        symbol = row['symbol']
        asset = symbol.replace('USDT', '')

        files = sorted(glob.glob(str(raw_dir / symbol / '*.csv')))
        if not files:
            continue

        df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
        
        # Auto-detect time format and filter valid range
        if 'open_time' in df.columns:
            sample_val = df['open_time'].iloc[0]
            if sample_val > 1e15:  # microseconds
                df['open_time'] = df['open_time'] // 1000  # convert to ms
            df = df[df['open_time'].between(1_600_000_000_000, 2_000_000_000_000)]
        
        df = df.drop_duplicates(subset='open_time').sort_values('open_time').reset_index(drop=True)

        if len(df) < 500:
            continue

        feat = build_features_fast(df)

        # Target: did price go up >2% in next 5 days (30 candles)?
        forward_ret = feat['_close'].shift(-TARGET_HORIZON) / feat['_close'] - 1
        feat['_target'] = (forward_ret > TARGET_THRESHOLD).astype(float)
        feat.loc[forward_ret.isna(), '_target'] = np.nan
        feat['_asset'] = asset
        feat['_ts'] = pd.to_datetime(df['open_time'], unit='ms')

        all_data.append(feat)

        if feature_cols is None:
            feature_cols = [c for c in feat.columns if not c.startswith('_')]

        if (idx + 1) % 25 == 0:
            print(f'  Loaded {idx + 1}/{len(manifest)} tokens...')

    combined = pd.concat(all_data, ignore_index=True)
    combined = combined.dropna(subset=feature_cols + ['_target']).reset_index(drop=True)

    print(f'\nTotal: {len(all_data)} tokens, {len(combined):,} samples, {len(feature_cols)} features')
    print(f'Date range: {combined["_ts"].min()} to {combined["_ts"].max()}')
    print(f'Target positive rate: {combined["_target"].mean()*100:.1f}%')
    return combined, feature_cols


# Load the data
print('Loading all tokens...')
t0 = time.time()
df_all, feature_cols = load_all_tokens(RAW_DIR, max_tokens=200)
print(f'Data loaded in {time.time()-t0:.0f}s')

In [ ]:
# ============================================================
# Cell 7: Data Exploration
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Feature distributions
sample = df_all[feature_cols].sample(min(5000, len(df_all)))
sample.hist(bins=50, ax=axes.flatten()[:4] if len(feature_cols) <= 4 else None, figsize=(16, 10))
plt.suptitle('Feature Distributions (sample)', fontsize=14)
plt.tight_layout()
plt.show()

# Correlation matrix
plt.figure(figsize=(14, 12))
corr = df_all[feature_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, fmt='.1f')
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

# Target distribution per token
target_by_token = df_all.groupby('_asset')['_target'].mean().sort_values(ascending=False)
plt.figure(figsize=(16, 5))
target_by_token.head(30).plot(kind='bar')
plt.title('Target Positive Rate by Token (top 30)', fontsize=14)
plt.ylabel('P(up >2% in 5d)')
plt.axhline(y=df_all['_target'].mean(), color='r', linestyle='--', label=f'Overall: {df_all["_target"].mean()*100:.1f}%')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Feature stats:')
print(df_all[feature_cols].describe().round(4).T)

---
## ML Tournament: Walk-Forward with 4 Models

Quarterly retraining, purged embargo gap, no lookahead bias.

In [ ]:
# ============================================================
# Cell 8: Baselines
# ============================================================

def run_baselines(df):
    """Run 3 non-ML baselines for comparison."""
    results = []

    # B0: Buy & Hold average across all tokens
    b0_rets = []
    for asset, g in df.groupby('_asset'):
        g = g.sort_values('_ts')
        b0_rets.append((g['_close'].iloc[-1] / g['_close'].iloc[0] - 1) * 100)
    results.append({'model': 'B0_BuyHold', 'total_return': np.mean(b0_rets),
                     'trades': 0, 'win_rate': 0, 'sharpe': 0, 'avg_return': np.mean(b0_rets)})

    # B1: SMA50 crossover
    above = df[df['above_sma50'] > 0.5]
    b1_ret = above['ret_6'].mean() * 100 * 42 if len(above) > 0 else 0
    results.append({'model': 'B1_SMA50', 'total_return': b1_ret,
                     'trades': len(above),
                     'win_rate': (above['ret_6'] > 0).mean() * 100 if len(above) > 0 else 0,
                     'sharpe': 0, 'avg_return': above['ret_6'].mean() * 100 if len(above) > 0 else 0})

    # B2: Momentum (top quartile by 7d return, weekly rebalance)
    b2_pnls = []
    timestamps = sorted(df['_ts'].unique())
    sample_points = timestamps[::42]  # weekly
    for i in range(len(sample_points) - 1):
        current = df[df['_ts'] == sample_points[i]]
        if len(current) < 20:
            continue
        top = current.nlargest(max(5, len(current) // 4), 'ret_42')
        for _, r in top.iterrows():
            future = df[(df['_asset'] == r['_asset']) & (df['_ts'] == sample_points[i + 1])]
            if not future.empty:
                pnl = (future.iloc[0]['_close'] / r['_close'] - 1) - FEE_PCT
                b2_pnls.append(pnl)

    b2_total = sum(b2_pnls) * 100 if b2_pnls else 0
    b2_wr = sum(1 for p in b2_pnls if p > 0) / len(b2_pnls) * 100 if b2_pnls else 0
    results.append({'model': 'B2_Momentum', 'total_return': b2_total,
                     'trades': len(b2_pnls), 'win_rate': b2_wr,
                     'sharpe': 0, 'avg_return': np.mean(b2_pnls) * 100 if b2_pnls else 0})

    return results


print('Running baselines...')
t0 = time.time()
baselines = run_baselines(df_all)
print(f'\nBaseline Results ({time.time()-t0:.0f}s):')
print(f'{"Model":<20} {"Trades":>7} {"Win%":>7} {"Total%":>9} {"Avg%":>8}')
print('-' * 55)
for b in baselines:
    print(f'{b["model"]:<20} {b["trades"]:>7} {b["win_rate"]:>6.1f}% {b["total_return"]:>+8.1f}% {b["avg_return"]:>+7.2f}%')

In [ ]:
# ============================================================
# Cell 9: ML Tournament — Walk-Forward
# ============================================================

def run_ml_model(df, feature_cols, model_factory, model_name):
    """Walk-forward for one ML model. Quarterly retraining with embargo."""
    df = df.sort_values('_ts').reset_index(drop=True)
    n = len(df)

    trade_pnls = []
    all_preds = []
    all_actuals = []
    all_probas = []
    retrains = 0

    cursor = MIN_TRAIN
    while cursor < n:
        train_end = cursor - 42  # 7-day embargo
        test_end = min(cursor + RETRAIN_EVERY, n)

        if train_end < 1000:
            cursor += RETRAIN_EVERY
            continue

        X_train = df[feature_cols].iloc[:train_end]
        y_train = df['_target'].iloc[:train_end]
        X_test = df[feature_cols].iloc[cursor:test_end]
        y_test = df['_target'].iloc[cursor:test_end]

        if len(X_test) == 0 or y_train.nunique() < 2:
            cursor += RETRAIN_EVERY
            continue

        t0 = time.time()
        model = model_factory()

        if model_name == 'M3_logistic':
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_train)
            X_te = scaler.transform(X_test)
            model.fit(X_tr, y_train)
            proba = model.predict_proba(X_te)[:, 1]
        else:
            model.fit(X_train, y_train)
            proba = model.predict_proba(X_test)[:, 1]

        preds = (proba >= CONFIDENCE_THRESHOLD).astype(int)
        all_preds.extend(preds.tolist())
        all_actuals.extend(y_test.tolist())
        all_probas.extend(proba.tolist())

        # Simulate trades
        for i, (pred, prob) in enumerate(zip(preds, proba)):
            if pred == 1 and prob >= CONFIDENCE_THRESHOLD:
                idx = cursor + i
                if idx + TARGET_HORIZON < n:
                    entry = df['_close'].iloc[idx]
                    exit_ = df['_close'].iloc[idx + TARGET_HORIZON]
                    pnl = (exit_ / entry - 1) - FEE_PCT
                    trade_pnls.append(pnl)

        retrains += 1
        elapsed = time.time() - t0
        print(f'    {model_name} retrain {retrains}: '
              f'train={len(X_train):,}, test={len(X_test):,}, '
              f'trades={len(trade_pnls)}, {elapsed:.1f}s')

        cursor += RETRAIN_EVERY

    # Compute metrics
    trades = len(trade_pnls)
    if trades > 0:
        wins = sum(1 for p in trade_pnls if p > 0)
        total_ret = sum(trade_pnls) * 100
        avg_ret = np.mean(trade_pnls) * 100
        sharpe = (np.mean(trade_pnls) / np.std(trade_pnls) * np.sqrt(365 / 5)
                  if np.std(trade_pnls) > 0 else 0)
        cum = np.cumsum(trade_pnls)
        peak = np.maximum.accumulate(cum)
        max_dd = float(np.min(cum - peak)) * 100
        accuracy = accuracy_score(all_actuals, all_preds) * 100
        try:
            auc = roc_auc_score(all_actuals, all_probas)
        except ValueError:
            auc = 0
    else:
        wins = total_ret = avg_ret = sharpe = max_dd = accuracy = auc = 0

    return {
        'model': model_name, 'retrains': retrains, 'trades': trades,
        'win_rate': wins / trades * 100 if trades > 0 else 0,
        'total_return': total_ret, 'avg_return': avg_ret,
        'sharpe': sharpe, 'max_dd': max_dd, 'accuracy': accuracy, 'auc': auc,
        'trade_pnls': trade_pnls,
    }


# Define model factories
models = {
    'M1_lightgbm': lambda: LGBMClassifier(
        n_estimators=100, max_depth=4, learning_rate=0.05,
        num_leaves=15, subsample=0.7, colsample_bytree=0.6,
        min_child_samples=50, is_unbalance=True,
        random_state=42, n_jobs=-1, verbose=-1,
    ),
    'M2_xgboost': lambda: XGBClassifier(
        n_estimators=100, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_weight=10,
        scale_pos_weight=2.0, random_state=42, n_jobs=-1, verbosity=0,
        eval_metric='logloss',
    ),
    'M3_logistic': lambda: LogisticRegression(
        C=0.1, max_iter=500, random_state=42,
    ),
    'M4_random_forest': lambda: RandomForestClassifier(
        n_estimators=100, max_depth=6, min_samples_leaf=50,
        max_features='sqrt', random_state=42, n_jobs=-1,
    ),
}

# Run tournament
print('=' * 70)
print('ML TOURNAMENT — Walk-Forward with Quarterly Retraining')
print('=' * 70)

ml_results = []
for name, factory in models.items():
    print(f'\n>>> {name}...')
    t0 = time.time()
    result = run_ml_model(df_all, feature_cols, factory, name)
    print(f'  {name} done in {time.time()-t0:.0f}s: '
          f'{result["trades"]} trades, wr={result["win_rate"]:.1f}%, '
          f'ret={result["total_return"]:+.1f}%, sharpe={result["sharpe"]:.2f}')
    ml_results.append(result)

In [ ]:
# ============================================================
# Cell 10: Tournament Results Summary
# ============================================================

print('\n' + '=' * 90)
print(f'TOURNAMENT RESULTS')
print(f'Data: {len(df_all):,} samples, {df_all["_asset"].nunique()} tokens, {len(feature_cols)} features')
print('=' * 90)

print(f'\n{"Model":<20} {"Trades":>7} {"Win%":>7} {"Total%":>9} {"Avg%":>8} {"Sharpe":>8} {"MaxDD%":>8} {"AUC":>6}')
print('-' * 85)

for b in baselines:
    print(f'{b["model"]:<20} {b["trades"]:>7} {b["win_rate"]:>6.1f}% '
          f'{b["total_return"]:>+8.1f}% {b["avg_return"]:>+7.2f}% {"n/a":>8} {"n/a":>8} {"n/a":>6}')

print('-' * 85)
ml_results.sort(key=lambda x: x['sharpe'], reverse=True)
for r in ml_results:
    print(f'{r["model"]:<20} {r["trades"]:>7} {r["win_rate"]:>6.1f}% '
          f'{r["total_return"]:>+8.1f}% {r["avg_return"]:>+7.2f}% '
          f'{r["sharpe"]:>7.2f} {r["max_dd"]:>+7.1f}% {r["auc"]:>5.3f}')

print('\n' + '=' * 90)
if ml_results:
    w = ml_results[0]
    print(f'WINNER: {w["model"]} (Sharpe={w["sharpe"]:.2f}, Return={w["total_return"]:+.1f}%, '
          f'WinRate={w["win_rate"]:.1f}%, AUC={w["auc"]:.3f})')
print('=' * 90)

In [ ]:
# ============================================================
# Cell 11: Tournament Visualization
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Equity curves
ax = axes[0, 0]
for r in ml_results:
    if r['trade_pnls']:
        cum_pnl = np.cumsum(r['trade_pnls']) * 100
        ax.plot(cum_pnl, label=f'{r["model"]} ({r["sharpe"]:.2f})')
ax.set_title('Cumulative P&L by Model (%)', fontsize=13)
ax.set_xlabel('Trade #')
ax.set_ylabel('Cumulative Return (%)')
ax.axhline(y=0, color='white', linestyle='--', alpha=0.3)
ax.legend(fontsize=9)

# 2. Win rates comparison
ax = axes[0, 1]
names = [r['model'] for r in ml_results]
win_rates = [r['win_rate'] for r in ml_results]
colors = ['#00ff88' if wr > 50 else '#ff4444' for wr in win_rates]
ax.barh(names, win_rates, color=colors)
ax.axvline(x=50, color='white', linestyle='--', alpha=0.5)
ax.set_title('Win Rate by Model (%)', fontsize=13)
ax.set_xlabel('Win Rate (%)')

# 3. Sharpe ratio comparison
ax = axes[1, 0]
sharpes = [r['sharpe'] for r in ml_results]
colors = ['#00ff88' if s > 0 else '#ff4444' for s in sharpes]
ax.barh(names, sharpes, color=colors)
ax.axvline(x=0, color='white', linestyle='--', alpha=0.5)
ax.set_title('Sharpe Ratio by Model', fontsize=13)
ax.set_xlabel('Sharpe Ratio')

# 4. Drawdown comparison
ax = axes[1, 1]
max_dds = [r['max_dd'] for r in ml_results]
ax.barh(names, max_dds, color='#ff6644')
ax.set_title('Max Drawdown by Model (%)', fontsize=13)
ax.set_xlabel('Max Drawdown (%)')

plt.suptitle('ML Tournament Results', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---
## Expert Backtest v3: BTC with Regime Filter

BTC-only backtest with:
- Classification target: P(BTC up >3% in 14d)
- 4-gate regime filter (trend, volatility, momentum, confidence)
- ATR-based stops with trailing stop
- XGBoost + LightGBM ensemble

In [ ]:
# ============================================================
# Cell 12: Load BTC Daily Data from 4h Candles
# ============================================================

def load_btc_daily(raw_dir):
    """Load BTC 4h candles and aggregate to daily."""
    btc_dir = raw_dir / 'BTCUSDT'
    files = sorted(glob.glob(str(btc_dir / '*.csv')))
    if not files:
        print('No BTC data found!')
        return None
    
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    
    # Auto-detect time format
    sample_val = df['open_time'].iloc[0]
    if sample_val > 1e15:
        df['open_time'] = df['open_time'] // 1000
    
    df['date'] = pd.to_datetime(df['open_time'], unit='ms').dt.date
    df['date'] = pd.to_datetime(df['date'])
    
    for col in ['open', 'high', 'low', 'close', 'volume']:
        df[col] = df[col].astype(float)
    
    # Aggregate to daily
    daily = df.groupby('date').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum',
    }).reset_index()
    
    daily = daily.sort_values('date').reset_index(drop=True)
    daily['asset'] = 'BTC'
    
    print(f'BTC daily: {len(daily)} rows ({daily.date.iloc[0].date()} -> {daily.date.iloc[-1].date()})')
    return daily


btc_daily = load_btc_daily(RAW_DIR)

In [ ]:
# ============================================================
# Cell 13: BTC Feature Engineering (Daily)
# ============================================================

def build_btc_features(daily):
    """Build stationary features for BTC daily data."""
    df = daily.copy()
    close = df['close']
    high = df['high']
    low = df['low']
    volume = df['volume']

    # Price-relative features
    for p in [10, 20, 50]:
        sma = close.rolling(p).mean()
        df[f'price_to_sma_{p}'] = close / sma - 1

    ema_12 = close.ewm(span=12, adjust=False).mean()
    ema_26 = close.ewm(span=26, adjust=False).mean()
    df['ema_12_26_ratio'] = ema_12 / ema_26 - 1

    # Momentum
    for d in [3, 7, 14, 28]:
        df[f'ret_{d}d'] = np.log(close / close.shift(d))

    # RSI
    delta = close.diff()
    gain = delta.where(delta > 0, 0.0).rolling(14).mean()
    loss = (-delta).where(delta < 0, 0.0).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    df['rsi_14'] = 100 - (100 / (1 + rs))

    # MACD
    macd = ema_12 - ema_26
    macd_signal = macd.ewm(span=9, adjust=False).mean()
    df['macd_hist_norm'] = (macd - macd_signal) / close

    # Bollinger
    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    df['bb_width'] = (2 * bb_std) / bb_mid
    df['bb_pct'] = (close - (bb_mid - 2 * bb_std)) / (4 * bb_std)

    # ATR
    prev_close = close.shift(1)
    tr = pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)
    atr_14 = tr.rolling(14).mean()
    df['atr_norm'] = atr_14 / close

    # Volume
    vol_sma_20 = volume.rolling(20).mean()
    df['volume_ratio'] = volume / vol_sma_20
    df['volume_change_5d'] = np.log(volume.rolling(5).mean() / vol_sma_20)

    # Volatility
    daily_ret = close.pct_change()
    df['volatility_10d'] = daily_ret.rolling(10).std()
    df['volatility_30d'] = daily_ret.rolling(30).std()
    df['vol_ratio'] = df['volatility_10d'] / df['volatility_30d']

    # Regime
    sma_50 = close.rolling(50).mean()
    sma_200 = close.rolling(200).mean()
    df['trend_50'] = (close > sma_50).astype(float)
    df['trend_200'] = (close > sma_200).astype(float)
    df['sma_50_200_cross'] = (sma_50 > sma_200).astype(float)

    # Day of week
    df['day_of_week'] = df['date'].dt.dayofweek / 6.0

    # Raw values for regime filter
    df['raw_close'] = close
    df['raw_atr_14'] = atr_14
    df['raw_sma_50'] = sma_50
    df['roc_20'] = close.pct_change(20)

    feature_cols = [
        'price_to_sma_10', 'price_to_sma_20', 'price_to_sma_50',
        'ema_12_26_ratio',
        'ret_3d', 'ret_7d', 'ret_14d', 'ret_28d',
        'rsi_14', 'macd_hist_norm',
        'bb_width', 'bb_pct',
        'atr_norm',
        'volume_ratio', 'volume_change_5d',
        'volatility_10d', 'volatility_30d', 'vol_ratio',
        'trend_50', 'trend_200', 'sma_50_200_cross',
        'day_of_week',
    ]

    df = df.dropna(subset=feature_cols).reset_index(drop=True)
    print(f'BTC features: {len(feature_cols)} columns, {len(df)} samples after warmup')
    return df, feature_cols


btc_df, btc_feature_cols = build_btc_features(btc_daily)

In [ ]:
# ============================================================
# Cell 14: BTC Walk-Forward + Regime Filter + Backtest
# ============================================================

def btc_build_target(df):
    """Classification: did BTC go up >3% in next 14 days?"""
    forward_ret = df['raw_close'].shift(-BTC_TARGET_HORIZON) / df['raw_close'] - 1
    target = (forward_ret > BTC_TARGET_RETURN_PCT).astype(int)
    valid = ~forward_ret.isna()
    pct_pos = target[valid].mean() * 100
    print(f'Target: P(up >{BTC_TARGET_RETURN_PCT*100:.0f}% in {BTC_TARGET_HORIZON}d) — {pct_pos:.1f}% positive rate')
    return target, valid


def regime_filter(row):
    """4-gate regime filter."""
    reasons = []
    if row['raw_close'] < row['raw_sma_50']:
        reasons.append('below 50-SMA')
    stop_pct = ATR_MULTIPLIER * row['raw_atr_14'] / row['raw_close']
    if stop_pct > ATR_STOP_CEILING:
        reasons.append(f'vol too high ({stop_pct:.1%})')
    if row['roc_20'] < MOMENTUM_GATE:
        reasons.append(f'ROC20={row["roc_20"]:.1%}')
    return len(reasons) == 0, reasons


def btc_walk_forward(df, feature_cols, target, valid):
    """Purged walk-forward with XGBoost + LightGBM ensemble."""
    n = len(df)
    all_signals = []
    last_signal_idx = -999
    models_trained = 0

    current = MIN_TRAIN_BTC
    while current < n:
        train_end = current - EMBARGO_DAYS
        test_end = min(current + BTC_RETRAIN_EVERY, n)

        if train_end < 100:
            current += BTC_RETRAIN_EVERY
            continue

        train_mask = valid.iloc[:train_end]
        X_train = df[feature_cols].iloc[:train_end][train_mask]
        y_train = target.iloc[:train_end][train_mask]

        if len(X_train) < 80 or y_train.sum() < 10:
            current += BTC_RETRAIN_EVERY
            continue

        # Train XGBoost
        xgb = XGBClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.7,
            min_child_weight=5, reg_alpha=0.5, reg_lambda=2.0,
            scale_pos_weight=len(y_train[y_train==0]) / max(len(y_train[y_train==1]), 1),
            random_state=42, n_jobs=-1, verbosity=0, eval_metric='logloss',
        )
        xgb.fit(X_train, y_train)

        # Train LightGBM
        lgb = LGBMClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.03,
            num_leaves=15, subsample=0.7, colsample_bytree=0.6,
            min_child_samples=10, reg_alpha=1.0, reg_lambda=3.0,
            is_unbalance=True, random_state=123, n_jobs=-1, verbose=-1,
        )
        lgb.fit(X_train, y_train)

        models_trained += 1
        print(f'  Retrain {models_trained}: train={len(X_train)}, '
              f'test={test_end-current} ({df.date.iloc[current].date()} -> {df.date.iloc[min(test_end-1, n-1)].date()})')

        # Generate signals on test chunk
        for i in range(current, test_end):
            if not valid.iloc[i]:
                continue
            if (i - last_signal_idx) < COOLDOWN_DAYS:
                continue

            row = df.iloc[i]
            passes, reasons = regime_filter(row)
            if not passes:
                continue

            X_i = df[feature_cols].iloc[[i]]
            xgb_prob = xgb.predict_proba(X_i)[0][1]
            lgb_prob = lgb.predict_proba(X_i)[0][1]
            avg_prob = 0.55 * xgb_prob + 0.45 * lgb_prob

            if avg_prob < BTC_CONFIDENCE_GATE:
                continue

            atr = row['raw_atr_14']
            entry_price = row['raw_close']
            stop_pct = max(ATR_MULTIPLIER * atr / entry_price, ATR_STOP_FLOOR)

            all_signals.append({
                'idx': i, 'date': row['date'], 'asset': 'BTC', 'action': 'BUY',
                'confidence': avg_prob, 'suggested_hold_days': BTC_TARGET_HORIZON,
                'stop_loss_pct': -stop_pct * 100, 'entry_price': entry_price,
                'atr_stop_price': entry_price * (1 - stop_pct),
            })
            last_signal_idx = i

        current = test_end

    print(f'Walk-forward: {models_trained} retrains, {len(all_signals)} signals')
    return all_signals


def btc_backtest(daily, signals):
    """Custom backtest with ATR stops, trailing stops, time exits."""
    if not signals:
        return None

    prices = daily.set_index('date')[['open', 'high', 'low', 'close']].copy()
    open_positions = []

    for sig in signals:
        sig_date = sig['date']
        future = prices.index[prices.index > sig_date]
        if len(future) == 0:
            continue
        entry_date = future[0]
        entry_price = float(prices.loc[entry_date, 'open'])
        stop_price = entry_price * (1 + sig['stop_loss_pct'] / 100)

        active = [p for p in open_positions if p['exit_date'] is None]
        if len(active) >= MAX_POSITIONS:
            continue

        open_positions.append({
            'entry_date': entry_date, 'entry_price': entry_price,
            'stop_price': stop_price, 'max_price': entry_price,
            'trailing_active': False, 'exit_date': None,
            'exit_price': None, 'exit_reason': None, 'confidence': sig['confidence'],
        })

    all_dates = sorted(prices.index)
    for date in all_dates:
        for pos in open_positions:
            if pos['exit_date'] is not None or date <= pos['entry_date']:
                continue

            day_high = float(prices.loc[date, 'high'])
            day_low = float(prices.loc[date, 'low'])
            day_close = float(prices.loc[date, 'close'])
            days_held = (date - pos['entry_date']).days

            if day_high > pos['max_price']:
                pos['max_price'] = day_high

            unrealized = (pos['max_price'] - pos['entry_price']) / pos['entry_price']
            if unrealized >= 0.08:
                pos['trailing_active'] = True
                pos['stop_price'] = max(pos['stop_price'], pos['max_price'] * 0.94)

            if day_low <= pos['stop_price']:
                pos['exit_date'] = date
                pos['exit_price'] = pos['stop_price']
                pos['exit_reason'] = 'trailing_stop' if pos['trailing_active'] else 'stop_loss'
                continue

            pnl_pct = (day_close - pos['entry_price']) / pos['entry_price']
            if days_held >= STAGNATION_DAYS and abs(pnl_pct) < STAGNATION_BAND:
                pos['exit_date'] = date
                pos['exit_price'] = day_close
                pos['exit_reason'] = 'stagnation'
                continue

            if days_held >= MAX_HOLD_DAYS:
                pos['exit_date'] = date
                pos['exit_price'] = day_close
                pos['exit_reason'] = 'max_hold'
                continue

    last_close = float(prices.iloc[-1]['close'])
    for pos in open_positions:
        if pos['exit_date'] is None:
            pos['exit_date'] = all_dates[-1]
            pos['exit_price'] = last_close
            pos['exit_reason'] = 'end_of_data'

    trades = []
    for pos in open_positions:
        pnl_pct = (pos['exit_price'] - pos['entry_price']) / pos['entry_price'] * 100
        trades.append({
            'entry_date': pos['entry_date'], 'exit_date': pos['exit_date'],
            'entry_price': pos['entry_price'], 'exit_price': pos['exit_price'],
            'pnl_pct': pnl_pct, 'pnl_idr': int(1_000_000 * pnl_pct / 100),
            'hold_days': (pos['exit_date'] - pos['entry_date']).days,
            'exit_reason': pos['exit_reason'], 'confidence': pos['confidence'],
        })

    return trades


# Run BTC backtest
print('=' * 70)
print('BTC EXPERT BACKTEST v3')
print('=' * 70)

btc_target, btc_valid = btc_build_target(btc_df)
btc_signals = btc_walk_forward(btc_df, btc_feature_cols, btc_target, btc_valid)

if btc_signals:
    btc_trades = btc_backtest(btc_daily, btc_signals)
    print(f'\nGenerated {len(btc_trades)} trades')
else:
    btc_trades = []
    print('No signals generated.')

In [ ]:
# ============================================================
# Cell 15: BTC Backtest Results
# ============================================================

if btc_trades:
    total_pnl = sum(t['pnl_idr'] for t in btc_trades)
    total_invested = len(btc_trades) * 1_000_000
    total_ret = total_pnl / total_invested * 100 if total_invested > 0 else 0
    winners = [t for t in btc_trades if t['pnl_pct'] > 0]
    losers = [t for t in btc_trades if t['pnl_pct'] <= 0]
    win_rate = len(winners) / len(btc_trades)
    avg_win = np.mean([t['pnl_pct'] for t in winners]) if winners else 0
    avg_loss = np.mean([t['pnl_pct'] for t in losers]) if losers else 0
    rr_ratio = abs(avg_win / avg_loss) if avg_loss != 0 else 0
    rets = [t['pnl_pct'] for t in btc_trades]
    sharpe = np.mean(rets) / np.std(rets) * np.sqrt(26) if len(rets) > 1 and np.std(rets) > 0 else 0
    cum_pnl = np.cumsum([t['pnl_idr'] for t in btc_trades])
    peak = np.maximum.accumulate(cum_pnl)
    dd = np.where(peak > 0, (cum_pnl - peak) / peak * 100, 0)
    max_dd = float(np.min(dd)) if len(dd) > 0 else 0

    print('\n' + '=' * 70)
    print('BTC BACKTEST RESULTS — Expert-Informed v3')
    print(f'Target: P(BTC up >{BTC_TARGET_RETURN_PCT*100:.0f}% in {BTC_TARGET_HORIZON}d)')
    print(f'Gates: 50-SMA + ATR vol + ROC20 momentum + {BTC_CONFIDENCE_GATE:.0%} confidence')
    print(f'Stops: {ATR_MULTIPLIER}x ATR (floor {ATR_STOP_FLOOR:.0%}, ceiling {ATR_STOP_CEILING:.0%})')
    print('=' * 70)

    print(f'\n{"Total Trades:":<30} {len(btc_trades)}')
    print(f'{"Winners / Losers:":<30} {len(winners)} / {len(losers)}')
    print(f'{"Win Rate:":<30} {win_rate:.1%}')
    print(f'{"Total Return (IDR):":<30} Rp {total_pnl:+,.0f}')
    print(f'{"Total Return (%):":<30} {total_ret:+.2f}%')
    print(f'{"Avg Win:":<30} {avg_win:+.2f}%')
    print(f'{"Avg Loss:":<30} {avg_loss:+.2f}%')
    print(f'{"Reward/Risk Ratio:":<30} {rr_ratio:.2f}')
    print(f'{"Sharpe (annualized):":<30} {sharpe:.2f}')
    print(f'{"Max Drawdown:":<30} {max_dd:.2f}%')
    print(f'{"Avg Hold Days:":<30} {np.mean([t["hold_days"] for t in btc_trades]):.1f}')

    # Exit reasons
    reasons = {}
    for t in btc_trades:
        r = t['exit_reason']
        reasons.setdefault(r, {'count': 0, 'pnl': 0})
        reasons[r]['count'] += 1
        reasons[r]['pnl'] += t['pnl_idr']
    print(f'\nEXIT REASONS:')
    for r, v in sorted(reasons.items()):
        print(f'  {r:<20} {v["count"]:>3} trades, Rp {v["pnl"]:>+12,.0f}')

    # Trade log
    print(f'\nTRADE LOG:')
    print(f'{"Entry":<12} {"Exit":<12} {"Entry$":>10} {"Exit$":>10} {"P&L%":>8} {"Days":>5} {"Conf":>6} {"Reason":<15}')
    print('-' * 82)
    for t in btc_trades:
        ed = t['entry_date'].strftime('%Y-%m-%d')
        xd = t['exit_date'].strftime('%Y-%m-%d')
        print(f'{ed:<12} {xd:<12} ${t["entry_price"]:>9,.0f} ${t["exit_price"]:>9,.0f} '
              f'{t["pnl_pct"]:>+7.2f}% {t["hold_days"]:>5} {t["confidence"]:>5.0%} {t["exit_reason"]:<15}')

    print(f'\nBENCHMARK:')
    bh_ret = (btc_daily.close.iloc[-1] / btc_daily.close.iloc[0] - 1) * 100
    print(f'  Buy & Hold:      {bh_ret:+.1f}%')
    print(f'  This strategy:   {total_ret:+.1f}% return, {max_dd:.1f}% max DD')
else:
    print('No BTC trades to show.')

In [ ]:
# ============================================================
# Cell 16: BTC Backtest Visualization
# ============================================================

if btc_trades:
    fig, axes = plt.subplots(3, 1, figsize=(16, 14))

    # 1. BTC price with entry/exit markers
    ax = axes[0]
    ax.plot(btc_daily['date'], btc_daily['close'], color='#888888', alpha=0.7, linewidth=0.8)
    for t in btc_trades:
        color = '#00ff88' if t['pnl_pct'] > 0 else '#ff4444'
        ax.scatter(t['entry_date'], t['entry_price'], color='cyan', marker='^', s=80, zorder=5)
        ax.scatter(t['exit_date'], t['exit_price'], color=color, marker='v', s=80, zorder=5)
    ax.set_title('BTC Price with Trade Entries/Exits', fontsize=13)
    ax.set_ylabel('Price (USD)')

    # 2. Cumulative P&L
    ax = axes[1]
    cum_pnl = np.cumsum([t['pnl_pct'] for t in btc_trades])
    colors = ['#00ff88' if p > 0 else '#ff4444' for p in [t['pnl_pct'] for t in btc_trades]]
    ax.bar(range(len(btc_trades)), [t['pnl_pct'] for t in btc_trades], color=colors, alpha=0.7)
    ax2 = ax.twinx()
    ax2.plot(cum_pnl, color='white', linewidth=2, label='Cumulative')
    ax.set_title('Trade P&L (%) + Cumulative', fontsize=13)
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Individual P&L (%)')
    ax2.set_ylabel('Cumulative P&L (%)')

    # 3. Monthly P&L
    ax = axes[2]
    monthly = {}
    for t in btc_trades:
        m = t['exit_date'].strftime('%Y-%m')
        monthly.setdefault(m, 0)
        monthly[m] += t['pnl_idr']
    months = sorted(monthly.keys())
    vals = [monthly[m] / 1_000_000 * 100 for m in months]
    colors = ['#00ff88' if v > 0 else '#ff4444' for v in vals]
    ax.bar(months, vals, color=colors)
    ax.set_title('Monthly P&L (% of 1M IDR)', fontsize=13)
    ax.set_ylabel('Return (%)')
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()
else:
    print('No trades to visualize.')

---
## Feature Importance Analysis

In [ ]:
# ============================================================
# Cell 17: Feature Importance (Train final models on full data)
# ============================================================

# Train final models for feature importance analysis
df_sorted = df_all.sort_values('_ts').reset_index(drop=True)
X_full = df_sorted[feature_cols]
y_full = df_sorted['_target']

# XGBoost
xgb_final = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.7, min_child_weight=10,
    random_state=42, n_jobs=-1, verbosity=0, eval_metric='logloss',
)
xgb_final.fit(X_full, y_full)

# LightGBM
lgb_final = LGBMClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    num_leaves=15, subsample=0.7, colsample_bytree=0.6,
    min_child_samples=50, is_unbalance=True,
    random_state=42, n_jobs=-1, verbose=-1,
)
lgb_final.fit(X_full, y_full)

# Plot feature importance
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# XGBoost importance
xgb_imp = pd.Series(xgb_final.feature_importances_, index=feature_cols).sort_values(ascending=True)
axes[0].barh(xgb_imp.index, xgb_imp.values, color='#00aaff')
axes[0].set_title('XGBoost Feature Importance', fontsize=13)

# LightGBM importance
lgb_imp = pd.Series(lgb_final.feature_importances_, index=feature_cols).sort_values(ascending=True)
axes[1].barh(lgb_imp.index, lgb_imp.values, color='#88ff00')
axes[1].set_title('LightGBM Feature Importance', fontsize=13)

plt.suptitle('Feature Importance Analysis', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# Top features agreement
print('\nTop 10 features by model:')
print(f'{"Rank":<6} {"XGBoost":<25} {"LightGBM":<25}')
print('-' * 56)
xgb_top = xgb_imp.sort_values(ascending=False).index[:10]
lgb_top = lgb_imp.sort_values(ascending=False).index[:10]
for i in range(10):
    print(f'{i+1:<6} {xgb_top[i]:<25} {lgb_top[i]:<25}')

In [ ]:
# ============================================================
# Cell 18: Save Results to Drive
# ============================================================

import json
from datetime import datetime

results_dir = Path('/content/drive/MyDrive/AI-Finance/results')
results_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Save tournament results
tournament_summary = {
    'timestamp': timestamp,
    'data': {'samples': len(df_all), 'tokens': int(df_all['_asset'].nunique()), 'features': len(feature_cols)},
    'baselines': baselines,
    'ml_results': [{k: v for k, v in r.items() if k != 'trade_pnls'} for r in ml_results],
}

with open(results_dir / f'tournament_{timestamp}.json', 'w') as f:
    json.dump(tournament_summary, f, indent=2, default=str)

# Save BTC backtest trades
if btc_trades:
    btc_df_trades = pd.DataFrame(btc_trades)
    btc_df_trades.to_csv(results_dir / f'btc_trades_{timestamp}.csv', index=False)

print(f'Results saved to {results_dir}')
print(f'  - tournament_{timestamp}.json')
if btc_trades:
    print(f'  - btc_trades_{timestamp}.csv')